# Bo‘g‘imTahlil — modelni Google Colab’da o‘qitish

Bu daftar **bepul GPU** da ResNet-18 va EfficientNet-B0 modellarini haqiqiy tizza rentgen tasvirlarida o‘qitadi va
dasturga tayyor `model.pt` faylini beradi. 6,7 GB dataset **Google serveriga** yuklanadi — kompyuteringizga emas.

**Dataset:** Chen P. *Knee Osteoarthritis Severity Grading Dataset*, Mendeley Data, 2018, doi:10.17632/56rmx5bjcr.1
(Osteoarthritis Initiative asosida; tizzalar alohida kesilgan, KL0–KL4).

**Boshlashdan oldin:** menyu → *Runtime / Среда выполнения* → *Change runtime type* → **T4 GPU** → Save.

Keyin kataklarni yuqoridan pastga ketma-ket ishga tushiring (▶). Jami vaqt: ~40–90 daqiqa.

In [ ]:
# 1-qadam: GPU borligini tekshirish
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YO‘Q — Runtime → Change runtime type → T4 GPU ni tanlang")

## 2-qadam: dastur fayllarini yuklash
Quyidagi katakni ishga tushiring va **BogimTahlil-1.0-MVP.zip** faylini tanlang (kompyuteringizdagi dastur arxivi, ~2 MB).

In [ ]:
from google.colab import files
import zipfile, pathlib, os
up = files.upload()
name = next(iter(up))
zipfile.ZipFile(name).extractall("/content")
PROJECT = next(p for p in pathlib.Path("/content").rglob("start.py") if (p.parent / "ai").is_dir()).parent
os.chdir(PROJECT)
print("Loyiha:", PROJECT)

## 3-qadam: dataset havolasi
1. https://data.mendeley.com/datasets/56rmx5bjcr/1 sahifasini oching.
2. **KneeXrayData.zip** qatoridagi yuklash belgisiga (⤓) **sichqonchaning o‘ng tugmasi** → *Копировать адрес ссылки / Copy link address*.
3. Havolani pastdagi `DATA_URL` ga qo‘ying.

Agar arxiv Google Drive’da bo‘lsa: `DATA_URL = "/content/drive/MyDrive/KneeXrayData.zip"` va avval Drive’ni ulang
(`from google.colab import drive; drive.mount('/content/drive')`).

In [ ]:
DATA_URL = ""   # ← havolani shu yerga qo‘ying
EPOCHS = 30     # TT: 30–50; erta to‘xtatish (7 epoch) yoqilgan
assert DATA_URL, "DATA_URL bo‘sh — 3-qadamdagi ko‘rsatmaga qarang"

## 4-qadam: o‘qitish (eng uzoq qadam)
Dataset yuklanadi, ikkala model o‘qitiladi, validation bo‘yicha eng yaxshisi tanlanadi, test to‘plamida bir marta
baholanadi va `artifacts/model.pt` ga eksport qilinadi. Natijalar hech qanday qo‘lda tuzatishsiz hisobotga yoziladi.

In [ ]:
!pip install -q -r ai/requirements.txt
!python ai/colab/colab_pipeline.py --data "$DATA_URL" --epochs $EPOCHS --num-workers 2 --out-zip /content/bogimtahlil_model_natijalari.zip

## 5-qadam: natijani yuklab olish
`bogimtahlil_model_natijalari.zip` (~50 MB) kompyuteringizga yuklanadi. Ichida: `artifacts/model.pt`, tajriba natijalari
(`experiments/`), dataset hisoboti va **10 ta haqiqiy test rentgen tasviri** (`demo_images/`, har bir KL darajadan 2 tadan).

In [ ]:
from google.colab import files
files.download("/content/bogimtahlil_model_natijalari.zip")

### Yuklab olish boshlanmasa (zaxira usullar)
- **A usul:** chap paneldagi 📁 (Files) belgisi → `bogimtahlil_model_natijalari.zip` → sichqonchaning o‘ng tugmasi → **Download**.
- **B usul:** quyidagi katak faylni **Google Drive**’ingizga saqlaydi (ruxsat so‘raladi). So‘ng drive.google.com dan yuklab oling.

In [ ]:
from google.colab import drive
import shutil
drive.mount("/content/drive")
shutil.copy("/content/bogimtahlil_model_natijalari.zip", "/content/drive/MyDrive/bogimtahlil_model_natijalari.zip")
print("Saqlandi: Google Drive → MyDrive → bogimtahlil_model_natijalari.zip")

## Keyin kompyuteringizda
1. `bogimtahlil_model_natijalari.zip` ichidagini **bogimtahlil** papkasiga ustidan nusxalang (artifacts, experiments, data, demo_images).
2. `AI_ORNATISH.bat` ni bir marta ishga tushiring (PyTorch, ~250 MB).
3. `BOGIMTAHLIL.bat` → *Rentgen tahlili* → `demo_images` dagi tasvirni yuklang. *Sun’iy intellekt modeli* sahifasida
   haqiqiy metrikalar va grafiklar paydo bo‘ladi.